# Geometry-aware CDFMM versus MagTense

This accuracy-focused comparison uses 1,000 deterministic tiles on a 10 x 10 x 10 lattice (one million source-target pairs). `FAST_N` may be set for a small smoke run. Every CDFMM call constructs a true dense direct plan with an explicit target-to-source identity map; finite-geometry source policy intentionally ignores it and retains finite self terms, while identity maps suppress coincident self interactions only for effective point sources.

The public MagTense comparison covers finite rectangular-prism and tetrahedron sources observed at points. CDFMM additionally evaluates exact rectangular-prism → rectangular-prism and tetrahedron → tetrahedron target averaging, comparing each with its point-target result as a **physical-model impact**, not a numerical error. Finite-geometry self interactions are retained and checked explicitly.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import cdfmm
from magtense import magstatics

RUN_FULL = os.environ.get('RUN_FULL', '1').lower() not in {'0', 'false', 'no'}
FAST_N = int(os.environ.get('FAST_N', '0') or 0)
N_TILES = 1000 if RUN_FULL and FAST_N == 0 else (FAST_N or 64)
SIDE = 10.0e-9
SPACING = 3.0 * SIDE
PRISM = cdfmm.RectangularPrism(0.8 * SIDE, 1.1 * SIDE, 0.6 * SIDE)
TETRA_OFFSETS = np.asarray([
    [-0.25, -0.25, -0.25], [0.75, -0.25, -0.25],
    [-0.25, 0.75, -0.25], [-0.25, -0.25, 0.75],
], dtype=np.float64) * (0.8 * SIDE)
TETRA = cdfmm.Tetrahedron(TETRA_OFFSETS)

# The default is exactly (10, 10, 10); FAST_N takes the first deterministic
# points of a sufficiently large centred lattice.
lattice_side = 10 if N_TILES == 1000 else max(1, int(np.ceil(N_TILES ** (1.0 / 3.0))))
grid = np.indices((lattice_side,) * 3, dtype=np.float64).reshape(3, -1).T
grid -= 0.5 * (lattice_side - 1.0)
centres = np.ascontiguousarray(SPACING * grid[:N_TILES])

# A fixed trigonometric state avoids dependence on platform RNG streams.
phase = np.arange(N_TILES, dtype=np.float64)
magnetisations = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * phase),
    -3.0e5 + 2.0e5 * np.cos(0.11 * phase),
    4.0e5 * np.sin(0.07 * phase + 0.3),
))
prism_moments = PRISM.volume * magnetisations
tetra_moments = TETRA.volume * magnetisations
identities = np.arange(N_TILES, dtype=np.int32)

def run_dense(source_geometry, target_geometry, moments, source_sizes=(),
              target_sizes=(), source_tetrahedra=(),
              target_tetrahedra=(),
              source_model=cdfmm.SourceModel.EXACT_GEOMETRY,
              target_model=cdfmm.TargetModel.POINT,
              source_positions=None, target_positions=None):
    source_positions = centres if source_positions is None else np.asarray(source_positions)
    target_positions = centres if target_positions is None else np.asarray(target_positions)
    local_identities = np.arange(len(target_positions), dtype=np.int32)
    plan = cdfmm.DenseDirectPlan(
        source_positions, target_positions,
        source_geometry=source_geometry, target_geometry=target_geometry,
        source_sizes=list(source_sizes), target_sizes=list(target_sizes),
        source_tetrahedra=list(source_tetrahedra),
        target_tetrahedra=list(target_tetrahedra),
        source_model=source_model,
        target_model=target_model,
        # The fixed identity map suppresses self interactions only when
        # the effective dense source model is a point dipole.
        target_source_indices=local_identities.tolist(),
        static_precision='float64',
    )
    return np.asarray(plan.evaluate(
        moments, backend=cdfmm.DenseDirectBackend.PORTABLE))

def magtense_prisms():
    tiles = magstatics.Tiles(
        n=N_TILES, tile_type=2,
        size=[PRISM.hx, PRISM.hy, PRISM.hz],
        offset=np.asfortranarray(centres), rot=[0.0, 0.0, 0.0], M_rem=0.0,
    )
    tiles.M = np.asfortranarray(magnetisations)
    points = np.asfortranarray(centres)
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_tetrahedra():
    # MagTense's tetrahedral vertices are supplied as global (n, 3, 4)
    # coordinates.  Zero offsets prevent a second translation.
    global_vertices = centres[:, None, :] + TETRA_OFFSETS[None, :, :]
    vertices = np.asfortranarray(np.transpose(global_vertices, (0, 2, 1)))
    tiles = magstatics.Tiles(n=N_TILES, tile_type=5, vertices=vertices,
                             offset=np.zeros_like(centres), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations)
    points = np.asfortranarray(centres)
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_single_prism():
    tiles = magstatics.Tiles(n=1, tile_type=2,
                             size=[PRISM.hx, PRISM.hy, PRISM.hz],
                             offset=np.zeros((1, 3)), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations[:1])
    points = np.asfortranarray(np.zeros((1, 3)))
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_single_tetra():
    vertices = np.asfortranarray(TETRA_OFFSETS.T[None, :, :])
    tiles = magstatics.Tiles(n=1, tile_type=5, vertices=vertices,
                             offset=np.zeros((1, 3)), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations[:1])
    points = np.asfortranarray(np.zeros((1, 3)))
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def errors(actual, reference):
    delta = actual - reference
    absolute = np.linalg.norm(delta, axis=1)
    scale = np.linalg.norm(reference, axis=1)
    relative = absolute / np.maximum(scale, np.finfo(float).eps)
    return {
        'relative_l2': float(np.linalg.norm(delta) / max(np.linalg.norm(reference), np.finfo(float).eps)),
        'max_absolute': float(absolute.max()),
        'max_relative': float(relative.max()),
        'component_rms': np.sqrt(np.mean(delta * delta, axis=0)),
    }

def record(label, actual, reference):
    result = errors(actual, reference)
    result['label'] = label
    return result

print(f'{N_TILES} tiles; {N_TILES * N_TILES:,} source-target pairs; full={RUN_FULL}')

## Geometry and lattice setup

The left panel shows every tile centre and a representative subset of magnetisation directions. The right panel shows the two finite source shapes to scale relative to `SIDE`; each observation is located at the corresponding tile centre.

In [ ]:
fig = plt.figure(figsize=(12, 5), constrained_layout=True)
ax = fig.add_subplot(121, projection='3d')
xyz_nm = centres / 1.0e-9
colour = np.linalg.norm(magnetisations, axis=1)
points = ax.scatter(*xyz_nm.T, c=colour, s=16, cmap='viridis', alpha=0.8)
stride = max(1, N_TILES // 80)
directions = magnetisations[::stride] / np.linalg.norm(magnetisations[::stride], axis=1)[:, None]
ax.quiver(*xyz_nm[::stride].T, *directions.T, length=0.55 * SPACING / 1.0e-9,
          normalize=True, color='black', alpha=0.45, linewidth=0.6)
ax.set(xlabel='x [nm]', ylabel='y [nm]', zlabel='z [nm]', title=f'{N_TILES} tile centres')
fig.colorbar(points, ax=ax, shrink=0.65, label=r'$|M|$ [A/m]')

shape_ax = fig.add_subplot(122, projection='3d')
half = 0.5 * np.array([PRISM.hx, PRISM.hy, PRISM.hz]) / SIDE
corners = np.array([[sx, sy, sz] for sx in (-half[0], half[0])
                    for sy in (-half[1], half[1]) for sz in (-half[2], half[2])])
prism_faces = [[corners[i] for i in face] for face in
               ((0, 1, 3, 2), (4, 5, 7, 6), (0, 1, 5, 4),
                (2, 3, 7, 6), (0, 2, 6, 4), (1, 3, 7, 5))]
shape_ax.add_collection3d(Poly3DCollection(prism_faces, alpha=0.28,
                                             facecolor='tab:blue', edgecolor='tab:blue'))
tetra_vertices = TETRA_OFFSETS / SIDE + np.array([1.5, 0.0, 0.0])
tetra_faces = [[tetra_vertices[i] for i in face] for face in
               ((0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3))]
shape_ax.add_collection3d(Poly3DCollection(tetra_faces, alpha=0.35,
                                             facecolor='tab:orange', edgecolor='tab:orange'))
shape_ax.scatter([0.0, 1.5], [0.0, 0.0], [0.0, 0.0], c='black', s=22, label='representative point')
shape_ax.text(0.0, -0.75, -0.55, 'rectangular prism', ha='center')
shape_ax.text(1.5, -0.75, -0.55, 'tetrahedron', ha='center')
shape_ax.set(xlim=(-0.8, 2.3), ylim=(-0.8, 0.8), zlim=(-0.8, 0.8),
             xlabel='x / SIDE', ylabel='y / SIDE', zlabel='z / SIDE',
             title='Representative finite geometries')
shape_ax.set_box_aspect((3.1, 1.6, 1.6))
shape_ax.legend(loc='upper right')
plt.show()

## Case 1 — RectangularPrism → Point

CDFMM uses exact finite prism P2P with point targets. MagTense uses `tile_type=2` and point observations.

In [ ]:
H_prism_point = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM, cdfmm.TargetGeometry.POINT,
    prism_moments, source_sizes=[PRISM],
)
H_prism_point_mt = magtense_prisms()
case_prism_point = record('prism -> point', H_prism_point, H_prism_point_mt)
# A single anisotropic prism at its representative checks the finite self term.
self_cdfmm = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM, cdfmm.TargetGeometry.POINT,
    prism_moments[:1], source_sizes=[PRISM],
    source_positions=centres[:1], target_positions=centres[:1],
)
assert np.all(np.isfinite(self_cdfmm))
self_mt = magtense_single_prism()
assert np.allclose(self_cdfmm, self_mt, rtol=5.0e-5, atol=1.0e-12)
print(case_prism_point)

## Case 2 — RectangularPrism → RectangularPrism

CDFMM evaluates exact rectangular-prism target averaging. Public MagTense has no equivalent observation API, so the reference here is CDFMM's point-target result. The reported difference measures how much target averaging changes this setup; it is not a numerical error.

In [ ]:
H_prism_prism = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM,
    cdfmm.TargetGeometry.RECTANGULAR_PRISM, prism_moments,
    source_sizes=[PRISM], target_sizes=[PRISM],
    target_model=cdfmm.TargetModel.EXACT_GEOMETRY,
)
case_prism_target_impact = record(
    'prism target averaging impact (vs point target)', H_prism_prism, H_prism_point
)
assert np.all(np.isfinite(H_prism_prism))
print(case_prism_target_impact)

## Cases 3–4 — tetrahedral source and physical tetrahedral target

The four vertices in `TETRA` are centroid-relative offsets. Case 3 uses exact tetrahedron source → point target and compares against MagTense `tile_type=5`. Case 4 (`tetrahedron -> tetrahedron`) uses the same source and target tetrahedra with `SourceModel.EXACT_GEOMETRY` and `TargetModel.EXACT_GEOMETRY`, so the field is averaged over each finite target volume. Its difference from tetrahedron → point is a **physical target-averaging effect**, not a numerical error. The coincident source/target pair is retained as a finite self interaction and checked below.

In [ ]:
H_tetra_point = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON, cdfmm.TargetGeometry.POINT,
    tetra_moments, source_tetrahedra=[TETRA],
)
H_tetra_point_mt = magtense_tetrahedra()
case_tetra_point = record('tetrahedron -> point', H_tetra_point, H_tetra_point_mt)
assert np.all(np.isfinite(H_tetra_point))

H_tetra_tetra = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON,
    cdfmm.TargetGeometry.TETRAHEDRON,
    tetra_moments, source_tetrahedra=[TETRA], target_tetrahedra=[TETRA],
    source_model=cdfmm.SourceModel.EXACT_GEOMETRY,
    target_model=cdfmm.TargetModel.EXACT_GEOMETRY,
)
case_tetra_target_impact = record(
    'tetrahedron target averaging impact (vs point target)',
    H_tetra_tetra, H_tetra_point,
)
assert np.all(np.isfinite(H_tetra_tetra))

# The coincident tetrahedron pair is a finite volume self interaction.
self_tetra_tetra = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON,
    cdfmm.TargetGeometry.TETRAHEDRON,
    tetra_moments[:1], source_tetrahedra=[TETRA],
    target_tetrahedra=[TETRA],
    source_model=cdfmm.SourceModel.EXACT_GEOMETRY,
    target_model=cdfmm.TargetModel.EXACT_GEOMETRY,
    source_positions=centres[:1], target_positions=centres[:1],
)
assert np.all(np.isfinite(self_tetra_tetra))
assert np.linalg.norm(self_tetra_tetra) > 0.0

self_tetra_point = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON, cdfmm.TargetGeometry.POINT,
    tetra_moments[:1], source_tetrahedra=[TETRA],
    source_positions=centres[:1], target_positions=centres[:1],
)
assert np.all(np.isfinite(self_tetra_point))
self_tetra_mt = magtense_single_tetra()
assert np.allclose(self_tetra_point, self_tetra_mt, rtol=5.0e-5, atol=1.0e-12)
print(case_tetra_point)
print(case_tetra_target_impact)

## Accuracy and model-impact summary

The MagTense rows are numerical accuracy checks. The CDFMM-only rows compare point targets with exact finite prism or tetrahedron target averaging and are reported separately as physical-model differences, not numerical errors.

In [ ]:
accuracy_rows = [case_prism_point, case_tetra_point]
model_rows = [case_prism_target_impact, case_tetra_target_impact]
print(f"{'comparison':52s} {'relative L2':>14s} {'max abs':>14s} {'max rel':>14s}")
for heading, group in (('MAGTENSE ACCURACY', accuracy_rows),
                       ('PHYSICAL MODEL / TARGET POLICY', model_rows)):
    print(f'-- {heading} --')
    for row in group:
        print(f"{row['label']:52s} {row['relative_l2']:14.6e} {row['max_absolute']:14.6e} {row['max_relative']:14.6e}")
assert all(np.isfinite(row['relative_l2']) for row in accuracy_rows + model_rows)
assert case_prism_point['relative_l2'] < 5.0e-5
assert case_tetra_point['relative_l2'] < 5.0e-5
print('PASS: both public-MagTense accuracy comparisons satisfy the smoke thresholds.')

## Comparison figures

Parity against MagTense is shown on the left. The right panel shows the per-target effects of exact rectangular-prism and tetrahedron target averaging relative to point targets in CDFMM. These are physical-model differences, not numerical errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for cdfmm_field, magtense_field, label, colour in (
    (H_prism_point, H_prism_point_mt, 'prism → point', 'tab:blue'),
    (H_tetra_point, H_tetra_point_mt, 'tetrahedron → point', 'tab:orange'),
):
    axes[0].scatter(np.linalg.norm(magtense_field, axis=1),
                    np.linalg.norm(cdfmm_field, axis=1), s=12, alpha=0.55,
                    label=label, color=colour)
limits = axes[0].get_xlim()
limits = (min(limits[0], axes[0].get_ylim()[0]), max(limits[1], axes[0].get_ylim()[1]))
axes[0].plot(limits, limits, 'k--', linewidth=1, label='ideal parity')
axes[0].set(xlim=limits, ylim=limits, xlabel=r'MagTense $|H|$ [A/m]',
            ylabel=r'CDFMM $|H|$ [A/m]', title='Public MagTense accuracy')
axes[0].legend()
axes[0].grid(alpha=0.2)

def per_target_relative_change(changed, baseline):
    return np.linalg.norm(changed - baseline, axis=1) / np.maximum(
        np.linalg.norm(baseline, axis=1), np.finfo(float).eps)

prism_change = np.sort(per_target_relative_change(H_prism_prism, H_prism_point))
tetra_change = np.sort(per_target_relative_change(H_tetra_tetra, H_tetra_point))
axes[1].plot(prism_change, label='prism averaging vs point', color='tab:blue')
axes[1].plot(tetra_change, label='tetrahedron averaging vs point', color='tab:orange')
axes[1].set(xlabel='target (sorted independently)', ylabel='relative field change',
            title='Target-model effect (not numerical error)', yscale='symlog',
            ylim=(-1.0e-17, None))
axes[1].grid(alpha=0.2)
axes[1].legend()
plt.show()